In [2]:
# with open('image.png', 'rb') as file:
#   response = ollama.chat(
#     model='tinyllama',
#     messages=[
#       {
#         'role': 'user',
#         'content': 'What is strange about this image?',
#         'images': [file.read()],
#       },
#     ],
#   )
# print(response['message']['content'])

In [3]:
# ollama.create?x

In [4]:
# modelfile='''
# FROM tinyllama
# SYSTEM You are a helpful chat assistant that uses the given information to do what the user has asked them to do.
# '''

# model = ollama.create(model='assistant-tinyllama', modelfile=modelfile)

In [5]:
# ollama.Client?

In [6]:
from ollama import chat, generate
from database import DatabaseClient

class ChatClient:

    def __init__(self, folder_path,  model = 'llava', transform_model = 'phi3'):
        self.messages = []
        self.model = model
        self.transform_model = transform_model
        self.database = DatabaseClient(folder_path)
        self.interact()
        self.database.close_connection()

    def transform_query(self, query):
        content = f"""Extract exactly 3-5 keywords from the following query, and return them in a comma-separated list with no additional text.

        Query: "{query}"

        Keywords: """

        transformed_query = generate(self.transform_model, prompt=content, options={'temperature':0})
        return transformed_query['response']
    
    def retrieve_from_query(self, user_content):
        transformed_query = self.transform_query(user_content)

        
        
    def retrieve_from_image(self, image_path):
        pass
        
    def interact(self):
        print("Type /exit to end.")
        while True:
            user_content = input("User: ")
            if user_content == "/exit":
                return
            self.messages.append({
                'role': 'user',
                'content': user_content
            })
            response = chat(model, stream=True, messages=self.messages, options={'temperature':0})
            assistant_content = ''
            for chunk in response:
                assistant_content += chunk['message']['content']
                print(chunk['message']['content'], end='', flush=True)
            print()
            print('-'*20)
            self.messages.append({
                    'role' : 'assistant',
                    'content' : assistant_content
                })




In [16]:

class ChatClient:

    def __init__(self, folder_path = "",  chat_model = 'llama3.1', transform_model = 'phi3'):
        self.messages = []
        self.chat_model = chat_model
        self.transform_model = transform_model
        self.database = DatabaseClient(folder_path=folder_path)
        
    
    def retriever(self, query = None, image_path = None):
        if image_path == None:
            return self.database.search_with_text(query)
        elif query == None:
            return self.database.search_with_image(image_path)
        else:
            return self.database.search_with_text(query).extend(self.database.search_with_image(image_path))
        

    def transform_query(self, query):
        content = f"""Extract exactly 3-5 keywords from the following query, and return them in a comma-separated list with no additional text.

        Query: "{query}"

        Keywords: """

        return generate(self.transform_model, prompt=content, options={'temperature':0})['response']
    
    def input_text(self, user_content):
        transfomed_query = self.transform_query(user_content)
        context = self.handle_properties(self.retriever(query = transfomed_query))
        return f"Given the {context} answer {user_content}"
    
        
    def handle_properties(self, properties):
        context = ''
        image_added = False
        for property in properties:
            if property['media_type'] == 'text':
                context += " " + property['text']
            if property['media_type'] == 'image' and not image_added:
                context += " " + property['path']
        return context

    
    
        
    def interact(self):
        print("Type /exit to end.")
        while True:
            user_content = input("User: ")
            if user_content == "/exit":
                self.database.close_connection()
                break
            self.messages.append({
                'role': 'user',
                'content': user_content
            })
            response = chat(self.chat_model, stream=True, messages=self.messages, options={'temperature':0})
            assistant_content = ''
            for chunk in response:
                assistant_content += chunk['message']['content']
                print(chunk['message']['content'], end='', flush=True)
            print()
            print('-'*20)
            self.messages.append({
                    'role' : 'assistant',
                    'content' : assistant_content
                })




In [17]:
chatbot = ChatClient()

In [15]:
chatbot.transform_query(query="What is the use of Machine Learningn in Economics?")

'Machine learning, economics, application, technology, data analysis'

In [134]:
chatbot.interact()

Type /exit to end.
The sky blue color is caused by the presence of various elements in the atmosphere, including:

1. Blue light: This is produced by the sun's ultraviolet rays that reach Earth's surface. The blue light is absorbed by oxygen molecules in the air, which causes the blue color to appear.

2. Water vapor: Water vapor is a major component of the atmosphere and contributes to the blue color of the sky. It absorbs some of the blue light that reaches Earth's surface.

3. Clouds: Clouds absorb some of the blue light, which then reflects back into space. This process creates the blue hue in the sky.

4. Sunlight: The sun's rays also contribute to the blue color of the sky. However, the amount of blue light absorbed by clouds and other factors can vary depending on the time of day, weather conditions, and location.
--------------------
The previous topic was "why is the sky blue?"
--------------------
